# 08 数据质检（每次产出后必跑）

**校验项**（溯源 + 一致性）:
1. 宽表 vs 明细的设备覆盖一致性
2. 退款字段口径: status 码 vs refund_apply_time 行为口径的偏差（已知线上缺口）
3. 社区标签 vs 明细可溯源（每个标签能在明细里找到证据单）
4. 订单量对账: 宽表 vs 明细
5. 社区图: device_community 与 merged 一致 + 产出完整性

> 产出: data_quality_report.csv，任何 FAIL 项都要人工确认后才能发布结果

In [1]:
import os, time
import numpy as np
import pandas as pd

BASE = os.environ.get("LEIDEN_BASE", os.path.abspath(os.path.join(os.getcwd(), "..")))
DATA = os.path.join(BASE, "data")
OUT  = os.path.join(BASE, "data", "model_output")

print("[数据质检] 开始")
report = []
def check(name, ok, detail):
    report.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}: {detail}")

import sys as _sys; _sys.path.insert(0, os.path.join(BASE, "tools"))
from data_loader import resolve_base, resolve_detail
base = pd.read_csv(resolve_base(), dtype=str)
det = pd.read_csv(resolve_detail(), dtype=str)
full = pd.read_csv(os.path.join(OUT, "final_merged_output.csv"), dtype=str,
                   usecols=["device_id", "community_id", "risk_level", "flight_refund_amount",
                            "flight_refund_order_cnt",
                            "flight_pay_ok_order_amount", "flight_total_order_cnt"])
comm = pd.read_csv(os.path.join(OUT, "device_community.csv"), dtype=str,
                   usecols=["node", "node_type", "community_id"])
tags = pd.read_csv(os.path.join(OUT, "community_risk_tags.csv"), dtype=str)

# 1. 设备覆盖
det_devs, base_devs = set(det["device_id"]), set(base["device_id"])
check("明细设备⊂宽表", det_devs.issubset(base_devs), f"明细 {len(det_devs)}, 不在宽表 {len(det_devs-base_devs)}")
graph_devs = set(comm[comm["node_type"]=="device"]["node"])
merged_devs = set(full[full["community_id"].astype(float) != -1]["device_id"])
_diff = len(graph_devs - merged_devs)
check("社区图=merged社区设备(容差2台:community_id为空的图设备)", _diff <= 2, f"图 {len(graph_devs)} vs merged {len(merged_devs)}, 差 {_diff} 台")

# 2. 退款口径（已知问题持续监控）
for c in ["create_time","pay_time","refund_apply_time"]:
    det[c] = pd.to_datetime(det[c], errors="coerce", format="mixed")
det["status_num"] = pd.to_numeric(det["status"], errors="coerce")
n_refund_behavior = det["refund_apply_time"].notna().sum()
check("退款行为单量(明细)", n_refund_behavior > 50000,
      f"refund_apply_time 非空 {n_refund_behavior} 单（status 码口径在宽表中已知不全, 溯源以明细为准）")
beh_not39 = det[det["refund_apply_time"].notna() & ~det["status_num"].isin([39,95,93,31,30])]
check("status码退款口径偏差(已知缺口,仅监控)", True,
      f"行为口径退款 {n_refund_behavior} 单, 其中 status 不在(39,95,93,31,30) 的 {len(beh_not39)} 单 ({len(beh_not39)/n_refund_behavior*100:.0f}%)")

# 3. 标签可溯源: 抽验「短时批量退款」社区——明细里该社区设备确实有退款行为
full["community_id"] = pd.to_numeric(full["community_id"], errors="coerce")
det_comm = det.merge(full[["device_id","community_id"]], on="device_id", how="left")
fast_comm = tags[tags["risk_tags"].fillna("").str.contains("短时批量退款")]["community_id"].astype(float)
sample_comms = list(fast_comm.head(5))
n_ok = 0
for cid in sample_comms:
    ev = det_comm[(det_comm["community_id"]==cid) & det_comm["refund_apply_time"].notna()]
    if len(ev) > 0: n_ok += 1
check("短时批量退款标签溯源", len(sample_comms)==0 or n_ok == len(sample_comms),
      f"抽验 {len(sample_comms)} 社区, {n_ok} 个在明细有退款证据单")

# 4. 订单量对账
full["order_cnt"] = pd.to_numeric(full["flight_total_order_cnt"], errors="coerce")
base_cnt = pd.to_numeric(base["flight_total_order_cnt"], errors="coerce")
check("宽表订单总量=base", abs(full["order_cnt"].sum() - base_cnt.sum()) < 1,
      f"merged {full['order_cnt'].sum():.0f} vs base {base_cnt.sum():.0f}")
det_cnt_by_dev = det.groupby("device_id")["order_no"].count()
base_idx = base.set_index("device_id")
common = det_cnt_by_dev.index.intersection(base_idx.index)
base_common = pd.to_numeric(base_idx.loc[common, "flight_total_order_cnt"], errors="coerce")
diff = (det_cnt_by_dev.loc[common] - base_common).abs()
ratio_ok = (diff / base_common.replace(0, np.nan) <= 0.05).mean()
check("明细订单量vs宽表(<=5%偏差)", ratio_ok >= 0.8,
      f"抽样 {len(common)} 设备, {ratio_ok*100:.0f}% 在 5% 偏差内")

# 4.5 退款率与金额一致性（社区级矛盾检测）
full["refund_cnt"] = pd.to_numeric(full["flight_refund_order_cnt"], errors="coerce").fillna(0)
full["refund_amt"] = pd.to_numeric(full["flight_refund_amount"], errors="coerce").fillna(0)
comm_check = full[full["community_id"] != -1].groupby("community_id").agg(
    rc=("refund_cnt", "sum"), ra=("refund_amt", "sum"))
conflict = comm_check[(comm_check["rc"] > 0) & (comm_check["ra"] == 0)]
n_conflict = len(conflict)
check("退款率>0但金额=0矛盾社区(线上支付表关联缺失)", n_conflict == 0,
      f"矛盾社区 {n_conflict} 个（根因: 支付表仅关联 33.9% 订单, 金额大面积缺失——前端已显示缺失而非 0; 需线上修复 SQL 关联口径）")

# 5. 产出完整性
for f in ["device_risk_score.csv","final_merged_output.csv","community_risk_tags.csv",
          "detail_device_features.csv","detail_device_features_v2.csv","machine_behavior_devices.csv"]:
    p = os.path.join(OUT, f)
    check(f"产出存在:{f}", os.path.exists(p), f"{os.path.getsize(p)/1024:.0f}KB" if os.path.exists(p) else "缺失")

rep = pd.DataFrame(report)
rep.to_csv(os.path.join(OUT, "data_quality_report.csv"), index=False, encoding="utf-8-sig")
n_fail = int((rep["status"]=="FAIL").sum())
print(f"\n[数据质检] 完成: {len(rep)} 项, FAIL {n_fail} 项")
print(f"报告: {os.path.join(OUT, 'data_quality_report.csv')}")

[数据质检] 开始
[data_loader] base 宽表: 26.08.27_base.csv


[data_loader] detail 明细: 26.08.27_detail.csv


  [PASS] 明细设备⊂宽表: 明细 21399, 不在宽表 0
  [PASS] 社区图=merged社区设备(容差2台:community_id为空的图设备): 图 24234 vs merged 24232, 差 2 台


  [PASS] 退款行为单量(明细): refund_apply_time 非空 77215 单（status 码口径在宽表中已知不全, 溯源以明细为准）
  [PASS] status码退款口径偏差(已知缺口,仅监控): 行为口径退款 77215 单, 其中 status 不在(39,95,93,31,30) 的 16982 单 (22%)


  [PASS] 短时批量退款标签溯源: 抽验 5 社区, 5 个在明细有退款证据单


  [PASS] 宽表订单总量=base: merged 8023768 vs base 8023768


  [PASS] 明细订单量vs宽表(<=5%偏差): 抽样 21399 设备, 97% 在 5% 偏差内


  [FAIL] 退款率>0但金额=0矛盾社区(线上支付表关联缺失): 矛盾社区 599 个（根因: 支付表仅关联 33.9% 订单, 金额大面积缺失——前端已显示缺失而非 0; 需线上修复 SQL 关联口径）
  [PASS] 产出存在:device_risk_score.csv: 149355KB
  [PASS] 产出存在:final_merged_output.csv: 371606KB
  [PASS] 产出存在:community_risk_tags.csv: 2854KB
  [PASS] 产出存在:detail_device_features.csv: 4694KB
  [PASS] 产出存在:detail_device_features_v2.csv: 2444KB
  [PASS] 产出存在:machine_behavior_devices.csv: 767KB

[数据质检] 完成: 14 项, FAIL 1 项
报告: /app/data/model_output/data_quality_report.csv
